# Interactive RAG with Docling + LangChain + FAISS

**Reference paper:** *Attention Is All You Need* (Vaswani et al., 2017)

This notebook walks through a complete Retrieval-Augmented Generation (RAG) pipeline:

1. Parse a PDF with **Docling** — extracting text, tables, figures, and formulas
2. Chunk content using **Docling HybridChunker** (structure-aware) for text
3. Attach **rich metadata** to every chunk (type, page, section, caption, bbox)
4. Embed and store all chunks in a **FAISS** vector store via **LangChain**
5. Retrieve with **MMR** (Maximal Marginal Relevance) for diverse results
6. Answer questions using an **LCEL** chain with **GPT-4o**

## 1. Setup & Dependencies

All packages are already installed in the project virtual environment. To install from scratch:

```bash
pip install docling langchain langchain-openai langchain-community faiss-cpu python-dotenv jupyter
```

In [ ]:
import os
import warnings
from pathlib import Path

from dotenv import load_dotenv

# Suppress minor deprecation noise
warnings.filterwarnings("ignore", category=DeprecationWarning)

# HybridChunker uses sentence-transformers tokenizer internally (for token counting only).
# We use OpenAI embeddings, so the ST model is never actually run.
# These two warnings are safe to suppress:

# 1) HF Hub unauthenticated access warning
os.environ["HF_HUB_VERBOSITY"] = "error"

# 2) "Token indices sequence length is longer than 512" — benign because:
#    - HybridChunker uses the ST tokenizer only for chunk-size decisions
#    - Long chunks (e.g. formulas) cannot be split further; they still embed fine via ada-002 (max 8,191 tokens)
warnings.filterwarnings(
    "ignore",
    message="Token indices sequence length is longer than the specified maximum"
)

load_dotenv()  # reads .env file
print("Environment loaded.")
print(f"OpenAI key set: {'OPENAI_API_KEY' in os.environ and bool(os.environ['OPENAI_API_KEY'])}")

## 2. Configuration

In [ ]:
PDF_PATH = Path("1706.03762v7.pdf")          # Attention Is All You Need
FAISS_INDEX_DIR = "faiss_nb_index"           # separate from the Streamlit app cache
EMBED_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-ada-002")
LLM_MODEL   = os.getenv("LLM_MODEL", "gpt-4o")

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
print(f"PDF   : {PDF_PATH}  ({PDF_PATH.stat().st_size // 1024} KB)")
print(f"Embed : {EMBED_MODEL}")
print(f"LLM   : {LLM_MODEL}")

## 3. PDF Parsing with Docling

Docling's `DocumentConverter` understands PDF structure deeply — it recovers:
- **Tables** with cell-level structure (exported to Markdown)
- **Figures / pictures** with their captions
- **Section headers** for hierarchical context
- **Body text** with reading-order preservation

> Parsing takes ~30-60 seconds on first run (model weights load once, then cache).

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions(
    do_table_structure=True,   # recover table cell structure
    do_ocr=False,              # text PDF — no OCR needed
    generate_picture_images=False,  # skip saving image crops (not needed for embeddings)
)

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

print("Converting PDF ...")
result = converter.convert(str(PDF_PATH))
doc = result.document

print(f"\nDocument stats:")
print(f"  Pages   : {len(doc.pages)}")
print(f"  Tables  : {len(doc.tables)}")
print(f"  Figures : {len(doc.pictures)}")

## 4. Metadata Design

Every chunk we store in FAISS carries this metadata schema. Rich metadata enables:
- **Filtered retrieval** — e.g., retrieve only tables, or only page 6
- **Transparent attribution** — show users where each answer came from
- **Context injection** — prepend `heading_path` to LLM context to orient the model

| Field | Type | Description |
|---|---|---|
| `source` | str | PDF filename |
| `type` | str | `text` \| `table` \| `figure` \| `formula` \| `section_header` |
| `page` | int | 1-indexed page number |
| `section` | str | Immediate parent heading |
| `heading_path` | str | `/`-separated ancestor headings (for display) |
| `caption` | str | Table or figure caption text |
| `bbox` | str | Bounding box as `l,b,r,t` in PDF points (bottom-left origin) |

> `heading_path` is stored as a flat string because FAISS metadata filters work on scalars.

## 5. Text Chunking with HybridChunker

Docling's `HybridChunker` splits the document into chunks that:
- Respect heading boundaries (no chunk spans two sections)
- Stay within a token budget (default: tokenizer max)
- Merge undersized consecutive chunks from the same section
- Automatically carry parent heading context in `chunk.meta.headings`

This is superior to generic character-based splitting for structured documents.

In [ ]:
from docling.chunking import HybridChunker
from langchain_core.documents import Document as LCDocument

chunker = HybridChunker()  # uses sentence-transformers tokenizer by default

text_chunks: list[LCDocument] = []

for chunk in chunker.chunk(doc):
    if not chunk.text.strip():
        continue

    meta = chunk.meta
    headings: list[str] = meta.headings or []  # already strings

    # Extract page number from first doc_item's provenance
    page_no = 0
    if meta.doc_items:
        item = meta.doc_items[0]
        if hasattr(item, "prov") and item.prov:
            page_no = int(item.prov[0].page_no)

    # Determine element type from label of first doc_item
    elem_type = "text"
    if meta.doc_items:
        raw_label = getattr(meta.doc_items[0], "label", None)
        label_str = raw_label.value if hasattr(raw_label, "value") else str(raw_label or "text")
        if label_str in ("section_header", "formula", "caption", "list_item"):
            elem_type = label_str

    text_chunks.append(LCDocument(
        page_content=chunk.text,
        metadata={
            "source": PDF_PATH.name,
            "type": elem_type,
            "page": page_no,
            "section": headings[-1] if headings else "",
            "heading_path": " / ".join(headings),
            "caption": "",
            "bbox": "",
        }
    ))

print(f"Text chunks: {len(text_chunks)}")

# Preview one chunk
sample = text_chunks[10]
print(f"\nSample chunk (index 10):")
print(f"  text     : {sample.page_content[:120]}...")
print(f"  page     : {sample.metadata['page']}")
print(f"  section  : {sample.metadata['section']}")
print(f"  heading_path: {sample.metadata['heading_path']}")

## 6. Table Extraction

**Strategy:** Each table becomes one chunk. Tables are exported to **Markdown** which preserves
cell structure and is readable by both the embedding model and the LLM.

The caption is stored in both `page_content` (so it's embedded and searchable) and `metadata['caption']` (for display in the UI).

In [ ]:
table_chunks: list[LCDocument] = []

for table in doc.tables:
    md_text = table.export_to_markdown(doc)   # pass doc to avoid deprecation warning
    caption = table.caption_text(doc) or ""

    page_no = 0
    bbox_str = ""
    if table.prov:
        prov = table.prov[0]
        page_no = int(prov.page_no)
        bb = prov.bbox
        bbox_str = f"{bb.l:.1f},{bb.b:.1f},{bb.r:.1f},{bb.t:.1f}"

    # Build content: caption first so semantic search surfaces tables via caption queries
    content_parts = []
    if caption:
        content_parts.append(caption)
    content_parts.append(f"\n{md_text}")
    content = "\n".join(content_parts)

    table_chunks.append(LCDocument(
        page_content=content,
        metadata={
            "source": PDF_PATH.name,
            "type": "table",
            "page": page_no,
            "section": "",
            "heading_path": "",
            "caption": caption,
            "bbox": bbox_str,
        }
    ))

print(f"Table chunks: {len(table_chunks)}")

# Preview
if table_chunks:
    t0 = table_chunks[0]
    print(f"\nSample table (page {t0.metadata['page']}):")
    print(f"  caption : {t0.metadata['caption'][:80]}...")
    print(f"  markdown (first 300 chars):")
    print(t0.page_content[:300])

## 7. Figure Extraction

**Strategy:** Caption + surrounding page context.

Since image pixels cannot be embedded in a text vector store, we embed:
1. The figure's **caption** (most semantically rich signal)
2. **Surrounding text** from the same page (up to 400 chars) for broader context

This lets semantic search find figures when users ask about what is depicted — e.g.,
*"show me the encoder-decoder architecture"* will surface Figure 1.

**Future upgrade:** Use a vision-language model (GPT-4o vision) to auto-generate figure
descriptions and embed those instead, using `pic.get_image(doc)` to get the image bytes.

In [ ]:
figure_chunks: list[LCDocument] = []

for pic in doc.pictures:
    caption = pic.caption_text(doc) or ""

    page_no = 0
    bbox_str = ""
    if pic.prov:
        prov = pic.prov[0]
        page_no = int(prov.page_no)
        bb = prov.bbox
        bbox_str = f"{bb.l:.1f},{bb.b:.1f},{bb.r:.1f},{bb.t:.1f}"

    # Gather surrounding text from same page for context
    page_texts = [
        c.page_content for c in text_chunks
        if c.metadata["page"] == page_no and c.metadata["type"] == "text"
    ]
    # Take first ~400 chars of nearest text block on that page
    surrounding = page_texts[0][:400] if page_texts else ""

    content_parts = []
    if caption:
        content_parts.append(caption)
    if surrounding:
        content_parts.append(f"Surrounding context: {surrounding}")
    if not content_parts:
        content_parts.append(f"[Figure on page {page_no}]")
    content = "\n\n".join(content_parts)

    figure_chunks.append(LCDocument(
        page_content=content,
        metadata={
            "source": PDF_PATH.name,
            "type": "figure",
            "page": page_no,
            "section": "",
            "heading_path": "",
            "caption": caption,
            "bbox": bbox_str,
        }
    ))

print(f"Figure chunks: {len(figure_chunks)}")

# Preview
if figure_chunks:
    f0 = figure_chunks[0]
    print(f"\nSample figure (page {f0.metadata['page']}):")
    print(f"  caption : {f0.metadata['caption']}")
    print(f"  content :\n{f0.page_content[:400]}")

## 8. Combine & Build FAISS Index

All chunk lists are merged, then embedded and indexed by LangChain's FAISS wrapper.

**Efficient metadata usage tip:** FAISS stores metadata per-vector as a Python dict in an
accompanying `.pkl` file. At retrieval time you can filter on any scalar metadata field
(e.g. `{"type": "table"}`) without re-embedding.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

all_docs = text_chunks + table_chunks + figure_chunks

print(f"Total chunks to embed:")
print(f"  text    : {len(text_chunks)}")
print(f"  tables  : {len(table_chunks)}")
print(f"  figures : {len(figure_chunks)}")
print(f"  TOTAL   : {len(all_docs)}")

# Embedding model — same as used by the Streamlit app
embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    openai_api_key=os.environ["OPENAI_API_KEY"],
)

print("\nBuilding FAISS index (calling OpenAI Embeddings API) ...")
vectorstore = FAISS.from_documents(all_docs, embeddings)

# Persist to disk
vectorstore.save_local(FAISS_INDEX_DIR)
print(f"\nFAISS index saved to '{FAISS_INDEX_DIR}/'")
print(f"  Files: {list(Path(FAISS_INDEX_DIR).iterdir())}")

## 9. Load Index & Configure Retriever

We demonstrate two retriever modes:

| Retriever | When to use |
|---|---|
| **MMR** (Maximal Marginal Relevance) | Default — returns diverse, non-redundant chunks |
| **Filtered** | When you know the content type (e.g., only tables) |

MMR is preferred for open-ended questions because the top-5 similarity results are often
near-duplicates. MMR re-ranks them to maximize coverage.

In [ ]:
# Load from disk (proves the persist/load roundtrip works)
vectorstore = FAISS.load_local(
    FAISS_INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True,  # safe: we wrote this file ourselves
)
print(f"Loaded vectorstore with {vectorstore.index.ntotal} vectors")

# MMR retriever — diverse results
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20},  # fetch 20, re-rank to top 5
)

# Filtered retriever — tables only (useful for data/comparison questions)
table_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3, "filter": {"type": "table"}},
)

# Quick smoke test
test_docs = retriever.invoke("multi-head attention")
print(f"\nRetriever smoke test — 'multi-head attention': {len(test_docs)} docs")
for d in test_docs:
    m = d.metadata
    print(f"  [{m['type']:8s}] page {m['page']:2d}  {d.page_content[:70]}...")

## 10. LCEL RAG Chain

The LangChain Expression Language (LCEL) pipeline:

```
question ──► retriever ──► format_docs ──┐
                                          ├──► prompt ──► LLM ──► answer
question ──────────────────────────────────┘
```

Key design choices:
- `format_docs` injects **type labels** and **page numbers** into the context so the LLM
  can reference them (e.g. "As shown in Table 1 on page 6...")
- The system prompt tells the LLM to **cite** tables and figures explicitly
- `temperature=0` for factual, deterministic answers

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs: list[LCDocument]) -> str:
    """Format retrieved chunks for injection into the LLM prompt.
    
    Each chunk is prefixed with a header showing its type, page, and caption
    so the LLM can cite sources precisely.
    """
    parts = []
    for doc in docs:
        m = doc.metadata
        elem_type = m.get("type", "text").upper()
        page = m.get("page", "?")
        caption = m.get("caption", "")
        section = m.get("section", "")

        header_parts = [f"{elem_type} | page {page}"]
        if section:
            header_parts.append(f"section: {section}")
        if caption:
            header_parts.append(f"caption: {caption[:80]}")
        header = "[" + "  |  ".join(header_parts) + "]"

        parts.append(f"{header}\n{doc.page_content}")

    return "\n\n" + "\n\n---\n\n".join(parts)


SYSTEM_PROMPT = """\
You are an expert on the research paper "Attention Is All You Need" (Vaswani et al., 2017).

Answer the user's question using ONLY the provided context chunks below.
Each chunk is labeled with its type (TEXT, TABLE, FIGURE) and page number.

Guidelines:
- Cite specific pages, tables, and figures by their labels when relevant.
- If a TABLE chunk contains the answer, quote or summarize it directly.
- If a FIGURE chunk is relevant, reference it by caption.
- If the context does not contain the answer, say "I don't have enough context to answer this."
- Be concise and precise.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0,
    openai_api_key=os.environ["OPENAI_API_KEY"],
)

# LCEL chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built.")
print("Chain: question -> retriever -> format_docs -> prompt -> GPT-4o -> answer")

## 11. Demo Q&A with Source Attribution

The `ask()` helper runs the full chain and also surfaces which chunks were retrieved,
with their type, page, and caption — full transparency into what the model used.

In [ ]:
def ask(question: str, use_retriever=None) -> str:
    """Run the RAG chain and print the answer + source attribution."""
    r = use_retriever or retriever
    print(f"{'='*70}")
    print(f"Q: {question}")
    print(f"{'='*70}")

    # Run chain
    if use_retriever:
        # Re-build chain with alternate retriever
        chain = (
            {"context": r | format_docs, "question": RunnablePassthrough()}
            | prompt | llm | StrOutputParser()
        )
        answer = chain.invoke(question)
    else:
        answer = rag_chain.invoke(question)

    print(f"\nA: {answer}")

    # Show sources
    sources = r.invoke(question)
    print(f"\nSources retrieved ({len(sources)}):")
    for i, s in enumerate(sources):
        m = s.metadata
        cap = m.get('caption', '')[:60]
        cap_str = f" — {cap}" if cap else ""
        print(f"  [{i+1}] [{m['type']:8s}] page {m['page']:2d}{cap_str}")
    return answer

In [ ]:
# Question 1: General text question
_ = ask("What is the purpose of multi-head attention and how does it differ from single-head attention?")

In [ ]:
# Question 2: Data question — likely surfaces a TABLE chunk
_ = ask("What training hyperparameters (learning rate, batch size, steps) were used?")

In [ ]:
# Question 3: Architecture question — should surface the FIGURE chunk
_ = ask("Describe the encoder-decoder architecture shown in the paper's main diagram.")

In [ ]:
# Question 4: Comparison question — use the table-only filtered retriever
print("Using TABLE-ONLY retriever for comparison question:\n")
_ = ask(
    "Compare the complexity per layer of self-attention vs recurrent layers.",
    use_retriever=table_retriever
)

## 12. Advanced: Metadata-Guided Retrieval Strategies

Efficient metadata usage goes beyond simple filtering. Here are patterns you can build on:

### A. Page-scoped retrieval
If the user is looking at a specific page (e.g. in the Streamlit viewer), narrow the search:

```python
page_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5, "filter": {"page": 6}}
)
```

### B. Section-scoped retrieval
Ask only within a section the user selected:

```python
section_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5, "filter": {"section": "Training"}}
)
```

### C. Multi-type hybrid — tables + figures only
FAISS filter requires exact match on a single key. For OR logic, run two retrievers
and merge results:

```python
from langchain.retrievers import MergerRetriever

r_tables  = vectorstore.as_retriever(search_kwargs={"k": 2, "filter": {"type": "table"}})
r_figures = vectorstore.as_retriever(search_kwargs={"k": 2, "filter": {"type": "figure"}})
visual_retriever = MergerRetriever(retrievers=[r_tables, r_figures])
```

### D. Heading-path injection in prompt
Use `heading_path` metadata to tell the LLM which section each chunk came from:

```python
def format_with_breadcrumb(docs):
    parts = []
    for d in docs:
        path = d.metadata.get('heading_path', '')
        prefix = f"Section: {path}\n" if path else ""
        parts.append(prefix + d.page_content)
    return "\n\n".join(parts)
```

### E. Vision upgrade for figures
For richer figure embeddings, use `pic.get_image(doc)` to get the image and
describe it with GPT-4o vision, then embed the description:

```python
import base64
from openai import OpenAI

client = OpenAI()

for pic in doc.pictures:
    img = pic.get_image(doc)  # returns PIL Image or bytes
    if img is None:
        continue
    # Encode to base64
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()
    
    # Describe with GPT-4o
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": [
            {"type": "text", "text": "Describe this figure from an ML paper in detail."},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}
        ]}]
    )
    description = resp.choices[0].message.content
    # Now embed `description` instead of just the caption
```